In [ ]:
from ddi.data import build_human, downsample_train_negatives
from ddi.train import train_and_eval
from ddi.experiment import log_run

train, val = build_human()
tr = downsample_train_negatives(train, 5)
cfg = {'model_name':'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext',
       'marker_init':'mean','epochs':3,'lr':2e-5,'batch_size':32,'max_length':256,'seed':0}
metrics = train_and_eval(cfg, tr, val)

log_run({**cfg, "neg_ratio": 5, "dataset": "human"}, metrics, notes="first working baseline")

In [ ]:
from collections import Counter
from ddi.data import load_brat_docs, make_sentence_level

docs = load_brat_docs("Train")
counts = Counter()
for doc in docs:
    for sd in make_sentence_level(doc):
        ents = {e.id: e for e in sd.entities}
        for rel in sd.relations:
            a1, a2 = ents[rel.arguments['Arg1']], ents[rel.arguments['Arg2']]
            order = "arg1_first" if a1.locations.begin() < a2.locations.begin() else "arg2_first"
            counts[(rel.type, order)] += 1
for k, v in sorted(counts.items()):
    print(k, v)

In [ ]:
import itertools
from ddi.data import build_human, downsample_train_negatives
from ddi.train import train_and_eval
from ddi.experiment import log_run

train, val = build_human()

BASE = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256}

for marker_init in ["subword", "random", "mean"]:
    for neg_ratio in [2, 5, None]:
        tr = downsample_train_negatives(train, neg_ratio, seed=42)
        for seed in [0, 1, 2]:
            cfg = {**BASE, "marker_init": marker_init, "neg_ratio": neg_ratio,
                   "seed": seed, "dataset": "human"}
            m = train_and_eval(cfg, tr, val)
            log_run(cfg, m)
            print(f"{marker_init:8} neg={str(neg_ratio):4} seed={seed}  "
                  f"f1={m['micro_f1_pos']:.3f}")

In [ ]:
from ddi.experiment import load_runs
df = load_runs()
df["cfg.neg_ratio"] = df["cfg.neg_ratio"].fillna("None")
df.groupby(["cfg.marker_init","cfg.neg_ratio"])[["m.micro_f1_pos","m.micro_p_pos","m.micro_r_pos"]].agg(["mean","std"])

In [ ]:
from ddi.data import build_human
train, val = build_human()
print(train[1])

from collections import Counter
per_sent = Counter(r["sent_id"] for r in train)
print(per_sent.most_common(1))

In [ ]:
from ddi.manifest import write_dataset
val_id = write_dataset(val, provenance="human:val", seed=42, notes="frozen eval set")
print(val_id)  